# Notebook 08: Predictive Campaign Response Modeling
**Focus:** Out-of-time temporal validation, Logistic regression baseline, XGBoost, PR-AUC evaluation, and feature importance.


In [ ]:
import joblib
import pandas as pd
import duckdb
con = duckdb.connect('../data/analytical/retail_bi.duckdb', read_only=True)


### 1. Out-of-Time Predicted Response Distribution


In [ ]:
preds = con.execute('SELECT * FROM campaign_predictions').df()
preds.head()


In [ ]:
preds['pred_decile'] = pd.qcut(preds['predicted_response_prob'].rank(method='first', ascending=False), q=10, labels=False) + 1
dec_summary = preds.groupby('pred_decile').agg(
    total=('response', 'count'),
    actual_responders=('response', 'sum'),
    response_rate=('response', 'mean')
).reset_index()
dec_summary['lift_vs_baseline'] = dec_summary['response_rate'] / preds['response'].mean()
dec_summary
